# 📈 Predicción de la dirección de acciones — Pipeline multi-ticker

Notebook del proyecto final de la *Tecnicatura en Ciencia de Datos*.
Para cada ticker (**AAPL, AMD, META, MSFT, NVDA**) entrena un sistema que predice si el precio
**sube o baja** al día siguiente y evalúa una estrategia de trading sobre datos *forward* (out-of-sample).

**Enfoque:** dos modelos por régimen temporal (2000–2019 y 2020–2025) combinados por *weighted vote*,
con **calibración** de probabilidades y *stacking*, alimentados con indicadores técnicos, macro (FRED),
fundamentales (FMP) y **sentimiento de noticias con FinBERT**.

> Ver el [`README`](../README.md) para resultados, gráficos y el análisis de limitaciones.
> ⚠️ Proyecto educativo — **no** es asesoramiento financiero.

## 1. Pipeline de entrenamiento multi-ticker

Bloque principal end-to-end. Para cada ticker:
1. **Ingesta de datos** — precios y fundamentales de *FMP*, series macro de *FRED* y titulares de noticias (*Finnhub → Google News RSS → Yahoo*).
2. **Sentimiento** — los titulares se puntúan con **FinBERT** (`yiyanghkust/finbert-tone`) y se agregan a una señal diaria.
3. **Feature engineering** — indicadores técnicos (retornos, EMA, volatilidad, ADX) con *lag* para evitar *look-ahead* (target en `t+1`, features en `t`).
4. **Modelado por régimen** — un modelo para 2000–2019 y otro para 2020–2025, combinados por *weighted vote* + **calibración** + **stacking**, con umbrales optimizados por F1.
5. **Persistencia** — guarda `models/*.joblib`, `results/ensembles/*_ensemble.json` y `*_results_summary.json`.

In [ ]:
# =============================================
#  NVDA BULL/BEAR — Dos modelos por época + Ensemble + Calibración + Stacking
# =============================================
import os
import json
from datetime import datetime, UTC

import joblib
import numpy as np
import pandas as pd
import requests
import torch
import matplotlib.pyplot as plt

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    accuracy_score, precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# =============================================
#              CONFIG
# =============================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🧠 Device Torch: {device}")

print("🔄 Cargando FinBERT (esto puede tardar unos segundos)...")
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone", local_files_only=False)
model_sent = AutoModelForSequenceClassification.from_pretrained(
    "yiyanghkust/finbert-tone",
    local_files_only=False
).to(device)
print("✅ FinBERT cargado")

FRED_API_KEY = os.getenv("FRED_API_KEY", "")
FMP_API_KEY = os.getenv("FMP_API_KEY", "")
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "")
USE_GPU = torch.cuda.is_available()

# === Fechas clave: ENTRENAMIENTO hasta 30-set-2025; FORWARD del 1 set al 11-dic-2025 ===
END_TRAIN = "2025-08-31"
FWD_START = "2025-09-01"
FWD_END   = "2025-12-11"
SP500_TRAIN_CSV = "sp500.csv"        # S&P500 para entrenamiento (cortado a agosto-2025)
SP500_TEST_CSV  = "sp500_test.csv"   # S&P500 con días de set oct y nov

# =============================================
#  Utils
# =============================================
def _safe_pct_change(s: pd.Series, periods: int) -> pd.Series:
    return s.pct_change(periods)

def _lag_cols(df: pd.DataFrame, cols: list[str], lag: int = 1) -> pd.DataFrame:
    lagged = df[cols].shift(lag)
    lagged.columns = [f"{c}_lag{lag}" for c in cols]
    return lagged

def _best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    p, r, t = precision_recall_curve(y_true, y_prob)
    f1 = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
    if len(t) == 0:
        return 0.5
    best = int(np.nanargmax(f1[:-1]))
    return float(np.clip(t[best], 0.01, 0.99))

def _weighted_average(p1: np.ndarray, p2: np.ndarray, w1: float, w2: float) -> np.ndarray:
    s = (w1 + w2) if (w1 + w2) > 0 else 1.0
    return (w1 * p1 + w2 * p2) / s
import numpy as np

def build_dynamic_signal(
    proba_s: np.ndarray,
    adx_lag1: np.ndarray,
    thr_strong: float | None = None,
    thr_strong_low: float = 0.50,
    thr_strong_high: float = 0.60,
    thr_mid_low: float = 0.45,
    thr_mid_high: float = 0.70,
    thr_flat_low: float = 0.50,
    thr_flat_high: float = 0.80,
):
    # si te pasan thr_strong, lo mappeo al low (y dejo high como low+0.10 por defecto)
    if thr_strong is not None:
        thr_strong_low = float(thr_strong)
        thr_strong_high = max(thr_strong_high, thr_strong_low + 0.10)

    p = np.asarray(proba_s, dtype=float)
    adx = np.asarray(adx_lag1, dtype=float)

    pred = np.zeros(len(p), dtype=int)
    no_trade = np.zeros(len(p), dtype=int)

    strong = adx >= 50
    mid = (adx >= 25) & (adx < 50)
    flat = adx < 25

    # strong: 0 / zone / 1
    pred[strong & (p >= thr_strong_high)] = 1
    no_trade[strong & (p >= thr_strong_low) & (p < thr_strong_high)] = 1

    # mid
    pred[mid & (p >= thr_mid_high)] = 1
    no_trade[mid & (p >= thr_mid_low) & (p < thr_mid_high)] = 1

    # flat
    pred[flat & (p >= thr_flat_high)] = 1
    no_trade[flat & (p >= thr_flat_low) & (p < thr_flat_high)] = 1

    return pred, no_trade



# === Helpers de deduplicación ===
def _unique_cols(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame):
        return df
    return df.loc[:, ~df.columns.duplicated()].copy()

def _unique_list(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# =============================================
#  Transformadores que preservan nombres/shape
# =============================================
class ColumnAligner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.columns_ = list(X.columns) if isinstance(X, pd.DataFrame) else [f"f_{i}" for i in range(np.shape(X)[1])]
        return self
    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.columns_)
        return df.reindex(columns=self.columns_)

class DataFrameSimpleImputer(SimpleImputer):
    def fit(self, X, y=None):
        self._cols = list(X.columns)
        return super().fit(X, y)
    def transform(self, X):
        arr = super().transform(X)
        return pd.DataFrame(arr, index=X.index, columns=self._cols)

class DataFrameRobustScaler(RobustScaler):
    def fit(self, X, y=None):
        self._cols = list(X.columns)
        return super().fit(X, y)
    def transform(self, X):
        arr = super().transform(X)
        return pd.DataFrame(arr, index=X.index, columns=self._cols)

class OutlierClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower_q: float = 0.01, upper_q: float = 0.99):
        self.lower_q = float(lower_q); self.upper_q = float(upper_q)
    def fit(self, X, y=None):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)
        self.lower_ = X_df.quantile(self.lower_q)
        self.upper_ = X_df.quantile(self.upper_q)
        return self
    def transform(self, X):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.columns_)
        return X_df.clip(lower=self.lower_, upper=self.upper_, axis=1)

class CorrelationSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold: float = 0.90):
        self.threshold = float(threshold)
    def fit(self, X, y=None):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        corr = X_df.corr().abs().fillna(0.0)
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [c for c in upper.columns if (upper[c] > self.threshold).any()]
        self.keep_ = [c for c in X_df.columns if c not in to_drop]
        return self
    def transform(self, X):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.keep_)
        return X_df[self.keep_]
    def get_feature_names_out(self, input_features=None):
        return np.array(self.keep_)

class AllNaNToConstant(BaseEstimator, TransformerMixin):
    """Evita que el Imputer salte columnas 100% NaN (shape mismatch)."""
    def __init__(self, constant: float = 0.0):
        self.constant = constant
        self.all_nan_cols_: list[str] = []
    def fit(self, X, y=None):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.all_nan_cols_ = [c for c in X_df.columns if not X_df[c].notna().any()]
        return self
    def transform(self, X):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.all_nan_cols_)
        if self.all_nan_cols_:
            X_df = X_df.copy()
            X_df[self.all_nan_cols_] = X_df[self.all_nan_cols_].fillna(self.constant)
        return X_df

# =============================================
#  Descarga/transform de fundamentales + FRED (serie y snapshot)
# =============================================
def _fetch_fundamentals_series(ticker: str) -> pd.DataFrame:
    endpoints = {
        "marketcap": f"https://financialmodelingprep.com/stable/historical-market-capitalization?symbol={ticker}&apikey={FMP_API_KEY}",
        "income": f"https://financialmodelingprep.com/stable/income-statement?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
        "balance": f"https://financialmodelingprep.com/stable/balance-sheet-statement?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
        "cashflow": f"https://financialmodelingprep.com/stable/cash-flow-statement?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
        "keymetrics": f"https://financialmodelingprep.com/stable/key-metrics?symbol={ticker}&period=annual&apikey={FMP_API_KEY}",
        "income_growth": f"https://financialmodelingprep.com/stable/income-statement-growth?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
        "balance_growth": f"https://financialmodelingprep.com/stable/balance-sheet-statement-growth?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
        "cashflow_growth": f"https://financialmodelingprep.com/stable/cash-flow-statement-growth?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
        "financial_growth": f"https://financialmodelingprep.com/stable/financial-growth?symbol={ticker}&period=quarter&apikey={FMP_API_KEY}",
    }
    dfs = []
    for name, url in endpoints.items():
        try:
            data = requests.get(url, timeout=20).json()
            df = pd.DataFrame(data)
            if not df.empty:
                date_col = [c for c in df.columns if "date" in c.lower() or "period" in c.lower()]
                if date_col:
                    df["Date"] = pd.to_datetime(df[date_col[0]], errors="coerce")
                df = df.drop_duplicates(subset="Date").set_index("Date").sort_index()
                df.columns = [f"{name}_{c}" for c in df.columns]
                dfs.append(df)
        except Exception as e:
            print(f"⚠️ fundamentals {name}: {e}")
    if not dfs:
        return pd.DataFrame()
    out = pd.concat(dfs, axis=1).ffill().reset_index().rename(columns={"index":"Date"})
    return out

def _fetch_fred_series() -> pd.DataFrame:
    fred_ids = ["CPIAUCSL","UNRATE","GDPC1","PCE","INDPRO","PAYEMS","GNPCA","M2SL","HOUST"]
    def fred(series_id):
        try:
            url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json"
            r = requests.get(url, timeout=25).json()
            df = pd.DataFrame(r["observations"])[["date", "value"]]
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            return df.dropna().set_index("date").rename(columns={"value": series_id})
        except Exception:
            return pd.DataFrame()
    parts = [fred(fid) for fid in fred_ids]
    parts = [p for p in parts if not p.empty]
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, axis=1).ffill().reset_index().rename(columns={"date":"Date"})
    return out

def fundamentals_snapshot_at(asof_date: pd.Timestamp) -> pd.DataFrame:
    """Devuelve una fila (Date=asof_date) con el último valor conocido de cada fundamental/FRED hasta esa fecha."""
    fnds = _fetch_fundamentals_series("NVDA")
    fred = _fetch_fred_series()
    base = pd.DataFrame({"Date":[pd.to_datetime(asof_date).normalize()]})
    for block in [fnds, fred]:
        if block is not None and not block.empty:
            tmp = pd.merge_asof(base.sort_values("Date"), block.sort_values("Date"), on="Date", direction="backward")
            base = pd.merge(base, tmp, on="Date", how="left")
    return base.fillna(method="ffill").fillna(method="bfill")

# =============================================
#  Noticias (Finnhub → GNews RSS → Yahoo → dummy) + FinBERT
# =============================================
def _chunk_dates(dstart: pd.Timestamp, dend: pd.Timestamp, days: int = 365):
    cur = dstart
    while cur <= dend:
        nxt = min(cur + pd.Timedelta(days=days-1), dend)
        yield cur, nxt
        cur = nxt + pd.Timedelta(days=1)

def _rss_parse(url: str) -> list[dict]:
    try:
        import feedparser
        d = feedparser.parse(url)
        rows = []
        for e in d.entries:
            rows.append({
                "title": getattr(e, "title", "") or "",
                "summary": getattr(e, "summary", "") or getattr(e, "description", "") or "",
                "url": getattr(e, "link", "") or "",
                "published": getattr(e, "published", "") or getattr(e, "updated", "") or "",
                "publisher": getattr(e, "source", {}).get("title", "").lower() if hasattr(e, "source") else (getattr(e, "publisher", "") or getattr(e, "author", "")).lower()
            })
        return rows
    except Exception:
        try:
            import re
            r = requests.get(url, timeout=15); r.raise_for_status()
            text = r.text
            items = re.findall(r"<item>(.*?)</item>", text, flags=re.DOTALL|re.IGNORECASE)
            rows = []
            for it in items:
                def _tag(t):
                    m = re.search(fr"<{t}>(.*?)</{t}>", it, flags=re.DOTALL|re.IGNORECASE)
                    return re.sub("<.*?>", "", m.group(1)).strip() if m else ""
                rows.append({
                    "title": _tag("title"),
                    "summary": _tag("description"),
                    "url": _tag("link"),
                    "published": _tag("pubDate"),
                    "publisher": ""
                })
            return rows
        except Exception:
            return []

def _normalize_rows(rows: list[dict], src: str) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame(columns=["Date","title","summary","url","source","publisher"])
    out = []
    for r in rows:
        if src == "finnhub":
            dt = r.get("datetime")
            if isinstance(dt, (int,float)): dt = pd.to_datetime(dt, unit="s", errors="coerce")
            else: dt = pd.to_datetime(dt, errors="coerce")
            out.append({
                "Date": dt.normalize() if pd.notna(dt) else pd.NaT,
                "title": r.get("headline") or r.get("title") or "",
                "summary": r.get("summary") or "",
                "url": r.get("url") or "",
                "source": "finnhub",
                "publisher": (r.get("source") or "").lower()
            })
        else:
            dt = pd.to_datetime(r.get("published"), errors="coerce")
            out.append({
                "Date": dt.normalize() if pd.notna(dt) else pd.NaT,
                "title": r.get("title") or "",
                "summary": r.get("summary") or "",
                "url": r.get("url") or "",
                "source": src,
                "publisher": (r.get("publisher") or r.get("source") or "").lower()
            })
    df = pd.DataFrame(out).dropna(subset=["Date"]).reset_index(drop=True)
    if not df.empty:
        if "url" in df.columns:
            df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
        df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
    return df

def _finnhub_fetch(symbol: str, start: pd.Timestamp, end: pd.Timestamp, api_key: str) -> pd.DataFrame:
    base = "https://finnhub.io/api/v1/company-news"
    all_rows = []
    for s,e in _chunk_dates(start, end, days=30):
        try:
            resp = requests.get(base, params={"symbol":symbol,"from":s.date().isoformat(),"to":e.date().isoformat(),"token":api_key}, timeout=20)
            if resp.status_code == 200:
                rows = resp.json()
                if isinstance(rows, list) and rows:
                    all_rows.extend(rows)
        except Exception:
            pass
    return _normalize_rows(all_rows, "finnhub")

def _gnews_fetch_multi(queries: list[str], start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """Mejor esfuerzo: usa when:Nd con el total de días (si el motor respeta >365 genial; si no, traerá lo más reciente)."""
    from urllib.parse import quote_plus
    days = int((end - start).days) + 1
    frames = []
    for q in queries:
        q2 = quote_plus(q + f" when:{days}d")
        url = f"https://news.google.com/rss/search?q={q2}&hl=en-US&gl=US&ceid=US:en"
        rows = _rss_parse(url)
        for r in rows:
            r["published"] = str(r.get("published",""))
        frames.append(_normalize_rows(rows, "gnews"))
    if frames:
        df = pd.concat(frames, axis=0, ignore_index=True)
        if "url" in df.columns: df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
        df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
        # filtra exacto por ventana
        df = df[(df["Date"] >= start.normalize()) & (df["Date"] <= end.normalize())]
        return df
    return pd.DataFrame(columns=["Date","title","summary","url","source","publisher"])

def _yahoo_rss_nvda(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    url = "https://feeds.finance.yahoo.com/rss/2.0/headline?s=NVDA&region=US&lang=en-US"
    rows = _rss_parse(url)
    for r in rows:
        r["published"] = str(r.get("published",""))
    df = _normalize_rows(rows, "yahoo")
    if not df.empty:
        df = df[(df["Date"] >= start.normalize()) & (df["Date"] <= end.normalize())]
    return df

# Ponderación simple por fuente
SOURCE_WEIGHTS = {
    "reuters": 1.5, "bloomberg": 1.5, "wall street journal": 1.4, "wsj": 1.4,
    "financial times": 1.3, "ft": 1.3, "the verge": 1.1, "the information": 1.2,
    "yahoo": 0.9, "seeking alpha": 0.9, "investing.com": 0.9, "motley fool": 0.9,
}
def _weight_for_publisher(p: str) -> float:
    if not p: return 1.0
    p = p.lower()
    for k,v in SOURCE_WEIGHTS.items():
        if k in p: return float(v)
    return 1.0

def _score_finbert_daily_weighted(df_news: pd.DataFrame) -> pd.DataFrame:
    if df_news.empty:
        return pd.DataFrame(columns=["Date","neg","neu","pos","news_count","neg_ewm3","neu_ewm3","pos_ewm3","news_pos_interact"])
    texts = (df_news["title"].fillna("") + ". " + df_news["summary"].fillna("")).tolist()
    weights = df_news["publisher"].apply(_weight_for_publisher).to_numpy()
    batch = 16
    probs_all, w_all = [], []
    for i in range(0, len(texts), batch):
        chunk = texts[i:i+batch]
        w_chunk = weights[i:i+batch]
        try:
            inputs = tokenizer(chunk, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
            with torch.no_grad():
                outputs = model_sent(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
            probs_all.append(probs); w_all.append(w_chunk)
        except Exception:
            continue
    if not probs_all: 
        return pd.DataFrame(columns=["Date","neg","neu","pos","news_count","neg_ewm3","neu_ewm3","pos_ewm3","news_pos_interact"])
    arr = np.vstack(probs_all); wv = np.concatenate(w_all)
    df_scored = df_news.iloc[:arr.shape[0]].copy()
    df_scored[["neg","neu","pos"]] = arr
    df_scored["w"] = wv

    # agregación diaria ponderada
    def _wmean(g, col):
        a = g[col].to_numpy(); w = g["w"].to_numpy()
        s = np.sum(w); 
        return float(np.sum(a*w)/s) if s > 0 else float(np.mean(a))
    grouped = []
    for d, g in df_scored.groupby("Date"):
        grouped.append({
            "Date": d,
            "neg": _wmean(g, "neg"),
            "neu": _wmean(g, "neu"),
            "pos": _wmean(g, "pos"),
            "news_count": float(g.shape[0])
        })
    daily = pd.DataFrame(grouped).sort_values("Date").reset_index(drop=True)
    for c in ["neg","neu","pos"]:
        daily[f"{c}_ewm3"] = daily[c].ewm(span=3, adjust=False).mean()
    daily["news_pos_interact"] = daily["news_count"] * daily["pos"]
    return daily

def _collect_news_with_fallback(start: pd.Timestamp, end: pd.Timestamp) -> tuple[pd.DataFrame, str]:
    """Finnhub (si hay key) → Google News RSS multi-query (máximo posible) → Yahoo → Dummy."""
    used_source = "dummy"
    df_news_all = pd.DataFrame(columns=["Date","title","summary","url","source","publisher"])

    # 1) Finnhub (cubre todo el rango exacto por chunks)
    try:
        api_key = FINNHUB_API_KEY if FINNHUB_API_KEY else None
        if api_key:
            df_fh = _finnhub_fetch("NVDA", start, end, api_key)
            if not df_fh.empty:
                df_news_all = df_fh; used_source = "finnhub"
    except Exception as e:
        print("⚠️ Finnhub:", e)

    # 2) Google News RSS con ventana = días totales (mejor esfuerzo)
    if df_news_all.empty:
        try:
            queries = ["NVDA OR Nvidia", "NVIDIA Corporation", "Nvidia earnings"]
            df_gn = _gnews_fetch_multi(queries, start, end)
            if not df_gn.empty:
                df_news_all = df_gn; used_source = "gnews"
        except Exception as e:
            print("⚠️ GNews:", e)

    # 3) Yahoo Finance RSS
    if df_news_all.empty:
        try:
            df_yh = _yahoo_rss_nvda(start, end)
            if not df_yh.empty:
                df_news_all = df_yh; used_source = "yahoo"
        except Exception as e:
            print("⚠️ Yahoo RSS:", e)

    # 4) Dummy
    if df_news_all.empty:
        print("🧠 FinBERT dummy (fallback)…")
        samples = [
            f"NVDA strong demand for AI chips",
            f"NVDA faces competition in GPU and data center markets",
            f"NVDA market outlook remains positive",
            f"NVDA growth prospects show investor optimism",
        ]
        dates_dummy = pd.date_range(start=start, end=end, freq="7D")
        df_news_all = pd.DataFrame({
            "Date": dates_dummy, "title": samples*(len(dates_dummy)//len(samples)+1),
            "summary": "", "url": "", "source": "dummy", "publisher": "dummy"
        }).iloc[:len(dates_dummy)]
        used_source = "dummy"

    return df_news_all, used_source

# =============================================
#  Dataset (anti-leak; target t+1 | X_t lag1)
# =============================================
def build_dataset_all(ticker: str, start_date: str = "2000-01-01", end_date: str = END_TRAIN):
    print(f"📡 Descargando datos diarios de {ticker} desde FMP... {start_date} → {end_date}")
    url_price = (
        f"https://financialmodelingprep.com/api/v3/historical-price-full/"
        f"{ticker}?from={start_date}&to={end_date}&apikey={FMP_API_KEY}"
    )
    r = requests.get(url_price).json()
    if "historical" not in r:
        raise ValueError("❌ No se pudieron obtener precios históricos desde FMP.")
    df_prices = pd.DataFrame(r["historical"])
    df_prices["Date"] = pd.to_datetime(df_prices["date"], errors="coerce")
    df_prices = df_prices.sort_values("Date").reset_index(drop=True)
    df_prices = df_prices[(df_prices["Date"] >= pd.to_datetime(start_date)) & (df_prices["Date"] <= pd.to_datetime(end_date))]
    print(f"✅ {len(df_prices)} días {df_prices['Date'].min().date()} → {df_prices['Date'].max().date()}")

    print("🏦 Fundamentales FMP...")
    fundamentals = _fetch_fundamentals_series(ticker)
    if fundamentals is None or fundamentals.empty:
        fundamentals = pd.DataFrame({"Date": df_prices["Date"]})
    print(f"✅ Fundamentales: {fundamentals.shape[0]} filas, {fundamentals.shape[1]} cols")

    print("📊 S&P500 local (sp500.csv)...")
    with open(SP500_TRAIN_CSV, "r", encoding="utf-8-sig") as f:
        first = f.readline(); sep = ";" if first.count(";") > first.count(",") else ","
    sp500 = pd.read_csv(SP500_TRAIN_CSV, sep=sep, encoding="utf-8-sig")
    sp500.columns = [c.strip().lower() for c in sp500.columns]
    date_col = next((c for c in sp500.columns if "date" in c or "fecha" in c), None)
    close_col = next((c for c in sp500.columns if "close" in c or "cierre" in c), None)
    sp500 = sp500[[date_col, close_col]].rename(columns={date_col:"Date", close_col:"sp500_close"})
    sp500["Date"] = pd.to_datetime(sp500["Date"], errors="coerce", dayfirst=True)
    sp500["sp500_close"] = sp500["sp500_close"].astype(str).str.replace(",", ".").astype(float)
    sp500 = sp500.dropna().sort_values("Date").reset_index(drop=True)

    print("💵 Treasury (rf)...")
    tdata = requests.get(f"https://financialmodelingprep.com/stable/treasury-rates?apikey={FMP_API_KEY}").json()
    df_treasury = pd.DataFrame(tdata)
    df_treasury["Date"] = pd.to_datetime(df_treasury["date"], errors="coerce")
    df_treasury = df_treasury.sort_values("Date")
    ten_cols = [c for c in df_treasury.columns if "10" in c and ("year" in c.lower() or "y" in c.lower())]
    if ten_cols:
        df_treasury["rf"] = pd.to_numeric(df_treasury[ten_cols[0]], errors="coerce") / 100.0
    else:
        numc = df_treasury.select_dtypes(include=[np.number]).columns
        df_treasury["rf"] = (df_treasury[numc].mean(axis=1) / 100.0) if len(numc) else 0.0

    print("🔗 Merge total...")
    merged = pd.merge_asof(df_prices.sort_values("Date"), fundamentals.sort_values("Date"), on="Date", direction="backward")
    merged = pd.merge_asof(merged.sort_values("Date"), sp500.sort_values("Date"), on="Date", direction="backward")
    merged = pd.merge_asof(merged.sort_values("Date"), df_treasury[["Date","rf"]].sort_values("Date"), on="Date", direction="backward")

    print("📈 Indicadores técnicos...")
    merged["r_nvda"] = merged["close"].pct_change()
    merged["r_mkt"]  = merged["sp500_close"].pct_change()
    merged["excess_nvda"] = merged["r_nvda"] - merged["rf"] / 252.0
    merged["excess_mkt"]  = merged["r_mkt"]  - merged["rf"] / 252.0
    cov = merged["excess_nvda"].rolling(30).cov(merged["excess_mkt"])
    var = merged["excess_mkt"].rolling(30).var()
    merged["beta_30d"] = cov / var

    high, low, close = merged["high"], merged["low"], merged["close"]
    plus_dm = np.where((high - high.shift(1)) > (low.shift(1) - low), np.maximum(high - high.shift(1), 0), 0)
    minus_dm = np.where((low.shift(1) - low) > (high - high.shift(1)), np.maximum(low - low.shift(1), 0), 0)
    tr = pd.concat([(high - low), (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    tr14 = tr.rolling(14).sum()
    plus_dm14 = pd.Series(plus_dm).rolling(14).sum()
    minus_dm14 = pd.Series(minus_dm).rolling(14).sum()
    plus_di14 = 100 * (plus_dm14 / tr14)
    minus_di14 = 100 * (minus_dm14 / tr14)
    dx = (abs(plus_di14 - minus_di14) / (plus_di14 + minus_di14)) * 100
    merged["ADX_14"] = dx.rolling(14).mean()

    merged["PVT"] = (merged["close"].pct_change() * merged["volume"]).cumsum()
    merged["TR"] = tr.values
    merged["ATR_14"] = tr.rolling(14).mean().values

    # 📰 Noticias entrenamiento (máximo posible) en TODO el rango de entrenamiento
    print("📰 Buscando noticias para ENTRENAMIENTO (máxima cobertura)…")
    train_start, train_end = merged["Date"].min().normalize(), merged["Date"].max().normalize()
    news_train, src_train = _collect_news_with_fallback(train_start, train_end)
    daily_sent_train = _score_finbert_daily_weighted(news_train) if not news_train.empty else pd.DataFrame()
    if daily_sent_train.empty:
        daily_sent_train = pd.DataFrame({"Date": merged["Date"].copy(),
                                         "neg":0.0,"neu":1.0,"pos":0.0,"news_count":0.0,
                                         "neg_ewm3":0.0,"neu_ewm3":1.0,"pos_ewm3":0.0,"news_pos_interact":0.0})
    merged = pd.merge_asof(merged.sort_values("Date"), daily_sent_train.sort_values("Date"), on="Date", direction="backward")
    print(f"✅ Sentimiento integrado (train) desde: {src_train} — días con noticias: {int((merged['news_count']>0).sum())}")

    print("📈 Features + target (lag1, anti-leak)...")
    merged = merged.sort_values("Date").reset_index(drop=True)
    merged["ema10"] = merged["close"].ewm(span=10).mean()
    merged["ema20"] = merged["close"].ewm(span=20).mean()
    merged["ema40"] = merged["close"].ewm(span=40).mean() 
    merged["ema50"] = merged["close"].ewm(span=50).mean()
    merged["ema100"] = merged["close"].ewm(span=100).mean()
    merged["ema10_above_ema50"] = (merged["ema10"] > merged["ema50"]).astype(int)
    merged["ema20_above_ema50"] = (merged["ema20"] > merged["ema50"]).astype(int)

    # momentum
    merged["slope_10"] = merged["close"].diff(10)
    merged["slope_20"] = merged["close"].diff(20)
    merged["slope_50"] = merged["close"].diff(50)

    merged["roc_5"]  = _safe_pct_change(merged["close"], 5)
    merged["roc_10"] = _safe_pct_change(merged["close"], 10)
    merged["roc_20"] = _safe_pct_change(merged["close"], 20)
    merged["roc_50"] = _safe_pct_change(merged["close"], 50)
    ret = merged["close"].pct_change()
    merged["vol_10"] = ret.rolling(10).std()
    merged["vol_20"] = ret.rolling(20).std()
    merged["vol_50"] = ret.rolling(50).std()
    merged["ewma_vol"] = ret.ewm(span=20).std()
    merged["sp_slope_20"] = merged["sp500_close"].diff(20)

    # === target (EMA40 + slope_10)
    merged["bull_t"] = ((merged["slope_10"] > 0) & (merged["close"] > merged["ema40"])).astype(int)
    merged["target"] = merged["bull_t"].shift(-1)

    # Señal trimestral y codificación cíclica
    merged["quarter"] = merged["Date"].dt.quarter.astype("Int64")
    q_dummies = pd.get_dummies(merged["quarter"], prefix="q", dtype=int)
    merged = pd.concat([merged, q_dummies], axis=1)
    q = merged["quarter"].astype(float)
    merged["q_sin"] = np.sin(2 * np.pi * (q - 1) / 4)
    merged["q_cos"] = np.cos(2 * np.pi * (q - 1) / 4)

    # Features base
    raw_cols = [
        "open","high","low","close","volume","ema10","ema20","ema40","ema50","ema100",
        "ema10_above_ema50","ema20_above_ema50","slope_10","slope_20","slope_50",
        "roc_5","roc_10","roc_20","roc_50","vol_10","vol_20","vol_50","ewma_vol",
        "r_nvda","r_mkt","excess_nvda","excess_mkt","beta_30d","ADX_14","PVT","TR","ATR_14",
        "sp500_close","sp_slope_20","rf",
        # Noticias
        "neg","neu","pos","news_count","pos_ewm3","neg_ewm3","news_pos_interact",
        # Trimestre
        "quarter","q_1","q_2","q_3","q_4","q_sin","q_cos",
    ]
    raw_cols = _unique_list([c for c in raw_cols if c in merged.columns])

    # Lags anti-leak + lag flexible de pos_ewm3
    feats_lag1 = _lag_cols(merged, raw_cols, lag=1)
    pos_ewm3_lag2 = merged[["pos_ewm3"]].shift(2) if "pos_ewm3" in merged.columns else pd.DataFrame(index=merged.index)
    if not pos_ewm3_lag2.empty:
        pos_ewm3_lag2.columns = ["pos_ewm3_lag2"]

    macro_cols = [c for c in merged.columns if c not in (["Date","target","bull_t"] + raw_cols)]
    feats_macro_lag1 = _lag_cols(merged, macro_cols, lag=1) if macro_cols else pd.DataFrame(index=merged.index)

    X_full = pd.concat([feats_lag1, feats_macro_lag1, pos_ewm3_lag2], axis=1)
    X_full = _unique_cols(X_full)

    df = pd.concat([merged[["Date","target"]], X_full], axis=1)
    df = df.dropna(subset=["target"]).reset_index(drop=True)

    non_lag = [c for c in df.columns if c not in ["Date","target"] and not c.endswith("_lag1") and c != "pos_ewm3_lag2"]
    df = df.drop(columns=non_lag) if non_lag else df
    df = _unique_cols(df)

    print(f"✅ Dataset final: {df.shape[0]} filas, {df.shape[1]-2} features (lag1).")

    # Mini-EDA por trimestre 
    try:
        tmp = merged.copy()
        tmp["target_t"] = merged["bull_t"].shift(-1)
        eda_q = (
            tmp.groupby("quarter", dropna=True)["target_t"]
            .mean().rename("bull_rate_next_day")
            .to_frame().reset_index()
            .sort_values("quarter")
        )
        print("\n📊 Bull-rate (t+1) por trimestre:")
        print(eda_q.to_string(index=False))
    except Exception:
        pass
    return df

# =============================================
#   Purged + Embargo TimeSeries Split
# =============================================
class PurgedTimeSeriesSplit:
    def __init__(self, n_splits=5, purge=5, embargo=5):
        self.n_splits=n_splits; self.purge=purge; self.embargo=embargo
    def split(self, X: pd.DataFrame):
        n = len(X)
        fold = n // (self.n_splits + 1)
        for i in range(self.n_splits):
            test_start = (i+1)*fold
            train_end = max(0, test_start - self.purge)
            test_end = min(n, test_start + fold)
            yield np.arange(0, train_end), np.arange(test_start, test_end)
    def get_n_splits(self, X=None, y=None, groups=None): return self.n_splits

# =============================================
#  Modelos y pipeline
# =============================================
def build_models_dict():
    xgb_params = dict(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=3.0,
        objective="binary:logistic", eval_metric="logloss",
        random_state=42, n_jobs=-1, scale_pos_weight=0.9, tree_method="hist",
    )
    lgbm_params = dict(
        boosting_type="gbdt", num_leaves=48, max_depth=-1, learning_rate=0.03,
        n_estimators=600, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.2, reg_lambda=0.4, min_child_samples=25, scale_pos_weight=0.9,
        random_state=42, n_jobs=-1, verbosity=-1, device_type="gpu" if USE_GPU else "cpu",
    )
    return {
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=12, min_samples_split=10, min_samples_leaf=20,
            max_features="sqrt", bootstrap=True, criterion="gini",
            random_state=42, n_jobs=-1, class_weight="balanced_subsample",
        ),
        "XGBoost": XGBClassifier(**xgb_params),
        "Logistic": LogisticRegression(
            penalty="l2", C=0.5, class_weight="balanced", solver="lbfgs", max_iter=1000, random_state=42,
        ),
        "LightGBM": LGBMClassifier(**lgbm_params),
    }

def make_pipeline(model):
    return Pipeline(steps=[
        ("align", ColumnAligner()),
        ("allnan", AllNaNToConstant(constant=0.0)),
        ("imputer", DataFrameSimpleImputer(strategy="median")),
        ("winsor", OutlierClipper(0.01, 0.99)),
        ("scaler", DataFrameRobustScaler(with_centering=True, with_scaling=True)),
        ("selector", CorrelationSelector(threshold=0.90)),
        ("varth", VarianceThreshold(1e-12)),
        ("model", model),
    ])

# =============================================
#  Entrenamiento por periodo (con calibración isotónica + umbral por época)
# =============================================
def train_period_model(df_all: pd.DataFrame,ticker: str, start: str, end: str, label: str):
    print(f"\n🗓️ Entrenando periodo {label}: {start} → {end}")
    msk = (df_all["Date"] >= pd.to_datetime(start)) & (df_all["Date"] <= pd.to_datetime(end))
    df = df_all.loc[msk].reset_index(drop=True)
    assert len(df) > 0, f"Periodo {label} vacío"

    feats_all = [c for c in df.columns if c not in ["Date","target"]]
    feats_num = [c for c in feats_all if pd.api.types.is_numeric_dtype(df[c])]
    low_var = [c for c in feats_num if df[c].nunique() <= 1]
    high_na = [c for c in feats_num if df[c].isna().mean() > 0.5]
    base_feats = [c for c in feats_num if c not in low_var + high_na]

    X = df[base_feats].copy()
    y = df["target"].astype(int).values

    models = build_models_dict()
    tscv = PurgedTimeSeriesSplit(n_splits=5, purge=5, embargo=5)

    best_name, best_cv = None, -np.inf
    for name, base_model in tqdm(models.items(), desc=f"🔁 Modelos {label}"):
        f1s=[]
        for tr_idx, te_idx in tscv.split(X):
            X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]
            pipe = make_pipeline(clone(base_model))
            pipe.fit(X_tr, y_tr)
            pred = pipe.predict(X_te)
            f1s.append(f1_score(y_te, pred))
        avg = float(np.mean(f1s))
        print(f"   → {name:12s} F1-CV: {avg:.4f}")
        if avg > best_cv: best_cv, best_name = avg, name

    print(f"🏆 Mejor {label}: {best_name} (F1-CV={best_cv:.4f})")

    # Split interno para calibración + umbral
    split_inner = int(len(X) * 0.9)
    X_tr, y_tr = X.iloc[:split_inner], y[:split_inner]
    X_val, y_val = X.iloc[split_inner:], y[split_inner:]

    base_pipe = make_pipeline(models[best_name])
    base_pipe.fit(X_tr, y_tr)

    # Calibración isotónica sobre el pipeline ya entrenado
    calib = CalibratedClassifierCV(base_pipe, method="isotonic", cv="prefit")
    calib.fit(X_val, y_val)

    val_prob_cal = calib.predict_proba(X_val)[:, 1]
    thr = _best_f1_threshold(y_val, val_prob_cal)

    artifact = {
        "calibrator": calib,
        "features": _unique_list(base_feats),
        "threshold": float(thr),
        "cv_f1": float(best_cv),
        "period": [start, end],
        "model_name": best_name,
    }
    joblib.dump(artifact, f"{ticker}_{label}_model.joblib")
    print(f"💾 Guardado {ticker}_{label}_model.joblib (thr={thr:.3f})")

    return {
        "label": label, "artifact": artifact,
        "cv_best": best_cv, "features": base_feats, "dates": (start, end)
    }

# =============================================
#  Helpers de predicción calibrada
# =============================================
def _predict_proba_calibrated(df_features: pd.DataFrame, artifact: dict) -> np.ndarray:
    feats = _unique_list(list(artifact["features"]))
    Xf = _unique_cols(df_features)
    feats_final = [c for c in feats if c in Xf.columns]
    X = Xf.reindex(columns=feats_final, fill_value=np.nan)
    return artifact["calibrator"].predict_proba(X)[:, 1]

# =============================================
#  Ensemble + Stacking + Reportes
# =============================================
def train_two_models_and_ensemble(ticker="NVDA", start_date="2000-01-01"):
    df_all = build_dataset_all(ticker, start_date, END_TRAIN)

    p1 = train_period_model(df_all,ticker, "2000-01-01", "2019-12-31", "2000_2019")
    p2 = train_period_model(df_all,ticker, "2020-01-01", END_TRAIN, "2020_2025")

    art1, art2 = p1["artifact"], p2["artifact"]
    w1, w2 = p1["cv_best"], p2["cv_best"]

    X_all = df_all[[c for c in df_all.columns if c not in ["Date","target"]]].copy()
    X_all = _unique_cols(X_all)

    proba1 = _predict_proba_calibrated(X_all, art1)
    proba2 = _predict_proba_calibrated(X_all, art2)
    proba_w = _weighted_average(proba1, proba2, w1, w2)

    # Stacking basado en [proba1, proba2]
    y_all = df_all["target"].astype(int).values
    X_meta_all = np.c_[proba1, proba2]

    def _rolling_splits(n: int, n_splits: int = 5, purge: int = 5):
        fold = n // (n_splits + 1)
        for i in range(n_splits):
            test_start = (i + 1) * fold
            train_end = max(0, test_start - purge)
            test_end = min(n, test_start + fold)
            yield np.arange(0, train_end), np.arange(test_start, test_end)

    f1s_meta = []
    for tr_idx, te_idx in _rolling_splits(len(df_all), n_splits=5, purge=5):
        X_tr, X_te = X_meta_all[tr_idx], X_meta_all[te_idx]
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]
        meta = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
        meta.fit(X_tr, y_tr)
        pred_te = meta.predict(X_te)
        f1s_meta.append(f1_score(y_te, pred_te))
    print(f"🧱 Meta-Logistic F1-CV (rolling): {np.mean(f1s_meta):.4f}")

    split_inner = int(0.9 * len(df_all))
    X_tr_m, y_tr_m = X_meta_all[:split_inner], y_all[:split_inner]
    X_val_m, y_val_m = X_meta_all[split_inner:], y_all[split_inner:]
    meta = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
    meta.fit(X_tr_m, y_tr_m)
    val_prob_m = meta.predict_proba(X_val_m)[:, 1]
    thr_meta = _best_f1_threshold(y_val_m, val_prob_m)
    joblib.dump({"meta_clf": meta, "threshold": float(thr_meta)}, f"{ticker}_stack.joblib")
    print(f"💾 Guardado {ticker}_stack.joblib (thr_meta={thr_meta:.3f})")

    proba_stack = meta.predict_proba(X_meta_all)[:, 1]
    thr1, thr2 = float(art1["threshold"]), float(art2["threshold"])
    thr_ens = (w1 * thr1 + w2 * thr2) / (w1 + w2 if (w1 + w2) > 0 else 1.0)

    # Voto por periodo (umbral por época)
    dates = pd.to_datetime(df_all["Date"])
    start1, end1 = pd.to_datetime(art1["period"][0]), pd.to_datetime(art1["period"][1])
    start2, end2 = pd.to_datetime(art2["period"][0]), pd.to_datetime(art2["period"][1])
    pred_period = np.zeros_like(proba_stack, dtype=int)
    m1 = (dates >= start1) & (dates <= end1)
    m2 = (dates >= start2) & (dates <= end2)
    pred_period[m1] = (proba1[m1] >= thr1).astype(int)
    pred_period[m2] = (proba2[m2] >= thr2).astype(int)
    rest = ~(m1 | m2)
    pred_period[rest] = (proba_w[rest] >= (thr1+thr2)/2.0).astype(int)

    pred_weighted = (proba_w >= thr_ens).astype(int)
    pred_stack = (proba_stack >= thr_meta).astype(int)

    # Evaluación holdout (último 20%)
    split_ho = int(len(df_all) * 0.8)
    y_ho = y_all[split_ho:]
    d_ho = df_all["Date"].iloc[split_ho:]

    def _report(tag, y_true, pred):
        f1 = f1_score(y_true, pred); acc = accuracy_score(y_true, pred)
        print(f"🎯 {tag} HOLDOUT → F1={f1:.4f}, ACC={acc:.4f}")
        print(confusion_matrix(y_true, pred))
        print(classification_report(y_true, pred, digits=4))
        return f1, acc

    f1_p, acc_p = _report("Period-Vote", y_ho, pred_period[split_ho:])
    f1_w, acc_w = _report("Weighted",    y_ho, pred_weighted[split_ho:])
    f1_s, acc_s = _report("Stacking",    y_ho, pred_stack[split_ho:])

    ensemble_art = {
        "model_p1": f"{ticker}_2000_2019_model.joblib",
        "model_p2": f"{ticker}_2020_2025_model.joblib",
        "weights": {"p1": float(w1), "p2": float(w2)},
        "thresholds": {"p1": float(thr1), "p2": float(thr2), "weighted": float(thr_ens)}
    }
    with open(f"{ticker}_ensemble.json", "w", encoding="utf-8") as f:
        json.dump(ensemble_art, f, indent=2)
    print(f"💾 Ensemble guardado en: {ticker}_ensemble.json")


    with open(f"{ticker}_results_summary.json","w",encoding="utf-8") as f:
        json.dump({
            "period_vote": {"f1": float(f1_p), "acc": float(acc_p)},
            "weighted":    {"f1": float(f1_w), "acc": float(acc_w)},
            "stacking":    {"f1": float(f1_s), "acc": float(acc_s)}
        }, f, indent=2)
    print(f"💾 Resultados guardados en: {ticker}_results_summary.json")

    return {
        "period_vote": (f1_p, acc_p),
        "weighted": (f1_w, acc_w),
        "stacking": (f1_s, acc_s),
        "thr_ens": float(thr_ens),
        "thr_meta": float(thr_meta)
    }

# =============================================
#  Forward test integrado (1 set – 12 dic 2025)
#  - Usa snapshot de fundamentales/FRED y ffill
#  - Usa noticias FinBERT de set oct nov 2025
#  - Usa exactamente las features que cada período entrenó
# =============================================
# Archivo forward por defecto (puedes cambiar el nombre si quieres)
FORWARD_RESULTS_FILE_TEMPLATE = "{ticker}_forward_{start}_to_{end}_ADXdyn.csv"


def run_forward_eval_nov(ticker="NVDA"):
    """
    Forward test del 1-sep al 11-dic-2025 con:
      - Snapshot de fundamentales/FRED al END_TRAIN (31-ago) ffill.
      - Noticias + FinBERT en todo el rango forward.
      - Probabilidades de:
          * modelo periodo 2000–2019
          * modelo periodo 2020–2025
          * ensemble ponderado
          * stacking (meta-logistic)
      - Target reconstruido OOS (EMA40 + slope_10, t+1).
      - Regla DINÁMICA con ADX + zona muerta:


            strong = adx >= 50 tendencia fuerte
            mid = (adx >= 25) & (adx < 50) tendencia moderada
            flat = adx < 25 tendencia débil
    """

    from sklearn.metrics import (
        f1_score, accuracy_score, confusion_matrix, classification_report
    )

    # ======================
    # 1) Fechas y lookback
    # ======================
    fwd_start = pd.to_datetime(FWD_START)
    fwd_end   = pd.to_datetime(FWD_END)
    lookback = 200
    start_need = (pd.to_datetime(END_TRAIN) - pd.Timedelta(days=lookback)).normalize()

    # ======================
    # 2) Cargar artefactos
    # ======================
    art_p1 = joblib.load(f"{ticker}_2000_2019_model.joblib")
    art_p2 = joblib.load(f"{ticker}_2020_2025_model.joblib")
    stack_art = joblib.load(f"{ticker}_stack.joblib")

    with open(f"{ticker}_ensemble.json","r",encoding="utf-8") as f:
        ens_cfg = json.load(f)

    w1, w2 = ens_cfg["weights"]["p1"], ens_cfg["weights"]["p2"]
    thr1, thr2 = ens_cfg["thresholds"]["p1"], ens_cfg["thresholds"]["p2"]
    thr_weighted = ens_cfg["thresholds"]["weighted"]

    thr_meta = float(stack_art["threshold"])
    meta_clf = stack_art["meta_clf"]

    FEATS_P1 = pd.Index(art_p1["features"]).unique()
    FEATS_P2 = pd.Index(art_p2["features"]).unique()

    # ======================
    # 3) Datos base (precios, S&P, rf)
    # ======================
    def fetch_prices_fmp(ticker, start_dt, end_dt, api_key):
        url = (
            f"https://financialmodelingprep.com/api/v3/historical-price-full/"
            f"{ticker}?from={start_dt.date()}&to={end_dt.date()}&apikey={api_key}"
        )
        r = requests.get(url, timeout=25).json()
        if "historical" not in r:
            raise RuntimeError("No prices from FMP")
        dfp = pd.DataFrame(r["historical"])
        dfp["Date"] = pd.to_datetime(dfp["date"], errors="coerce")
        dfp = dfp.sort_values("Date").reset_index(drop=True)
        keep = ["Date", "open", "high", "low", "close", "volume"]
        for k in keep:
            if k not in dfp.columns and k != "Date":
                if k == "close" and "adjClose" in dfp.columns:
                    dfp["close"] = pd.to_numeric(dfp["adjClose"], errors="coerce")
                else:
                    dfp[k] = np.nan
        return dfp[keep]

    def load_sp500_csv(path):
        with open(path, "r", encoding="utf-8-sig") as f:
            first = f.readline()
            sep = ";" if first.count(";") > first.count(",") else ","
        sp = pd.read_csv(path, sep=sep, encoding="utf-8-sig")
        sp.columns = [c.strip().lower() for c in sp.columns]
        date_col  = next((c for c in sp.columns if "date" in c or "fecha" in c), None)
        close_col = next((c for c in sp.columns if "close" in c or "cierre" in c), None)
        sp = sp[[date_col, close_col]].rename(columns={date_col: "Date", close_col: "sp500_close"})
        sp["Date"] = pd.to_datetime(sp["Date"], errors="coerce", dayfirst=True)
        sp["sp500_close"] = pd.to_numeric(
            sp["sp500_close"].astype(str).str.replace(",", "."),
            errors="coerce"
        )
        sp = sp.dropna().sort_values("Date").reset_index(drop=True)
        return sp

    def fetch_treasury_10y(api_key):
        url = f"https://financialmodelingprep.com/stable/treasury-rates?apikey={api_key}"
        t = requests.get(url, timeout=25).json()
        dt = pd.DataFrame(t)
        dt["Date"] = pd.to_datetime(dt["date"], errors="coerce")
        dt = dt.sort_values("Date")
        ten_cols = [c for c in dt.columns if "10" in c and ("year" in c.lower() or "y" in c.lower())]
        if ten_cols:
            dt["rf"] = pd.to_numeric(dt[ten_cols[0]], errors="coerce") / 100.0
        else:
            numc = dt.select_dtypes(include=[np.number]).columns
            dt["rf"] = (dt[numc].mean(axis=1) / 100.0) if len(numc) else 0.0
        return dt[["Date", "rf"]]

    prices = fetch_prices_fmp(ticker, start_need, fwd_end, FMP_API_KEY)
    sp500  = load_sp500_csv(SP500_TEST_CSV)
    treas  = fetch_treasury_10y(FMP_API_KEY)

    merged = pd.merge_asof(prices.sort_values("Date"),
                           sp500.sort_values("Date"),
                           on="Date", direction="backward")
    merged = pd.merge_asof(merged.sort_values("Date"),
                           treas.sort_values("Date"),
                           on="Date", direction="backward")

    # ======================
    # 4) Técnicos (incluye ADX_14)
    # ======================
    ret = merged["close"].pct_change()
    merged["r_nvda"] = ret
    merged["r_mkt"]  = merged["sp500_close"].pct_change()
    merged["excess_nvda"] = merged["r_nvda"] - merged["rf"]/252.0
    merged["excess_mkt"]  = merged["r_mkt"]  - merged["rf"]/252.0

    high, low, close = merged["high"], merged["low"], merged["close"]
    tr = pd.concat([
        (high - low),
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    merged["TR"] = tr.values
    merged["ATR_14"] = tr.rolling(14).mean().values

    plus_dm = np.where(
        (high - high.shift(1)) > (low.shift(1) - low),
        np.maximum(high - high.shift(1), 0), 0
    )
    minus_dm = np.where(
        (low.shift(1) - low) > (high - high.shift(1)),
        np.maximum(low - low.shift(1), 0), 0
    )
    tr14 = tr.rolling(14).sum()
    plus_di14 = 100 * (pd.Series(plus_dm).rolling(14).sum() / tr14)
    minus_di14 = 100 * (pd.Series(minus_dm).rolling(14).sum() / tr14)
    dx = (abs(plus_di14 - minus_di14) / (plus_di14 + minus_di14)) * 100
    merged["ADX_14"] = dx.rolling(14).mean()

    merged["ema10"]  = merged["close"].ewm(span=10).mean()
    merged["ema20"]  = merged["close"].ewm(span=20).mean()
    merged["ema40"]  = merged["close"].ewm(span=40).mean()
    merged["ema50"]  = merged["close"].ewm(span=50).mean()
    merged["ema100"] = merged["close"].ewm(span=100).mean()
    merged["ema10_above_ema50"] = (merged["ema10"] > merged["ema50"]).astype(int)
    merged["ema20_above_ema50"] = (merged["ema20"] > merged["ema50"]).astype(int)

    merged["slope_10"] = merged["close"].diff(5)
    merged["slope_20"] = merged["close"].diff(20)
    merged["slope_50"] = merged["close"].diff(50)

    merged["roc_5"]  = _safe_pct_change(merged["close"], 5)
    merged["roc_10"] = _safe_pct_change(merged["close"],10)
    merged["roc_20"] = _safe_pct_change(merged["close"],20)
    merged["roc_50"] = _safe_pct_change(merged["close"],50)

    merged["vol_10"] = ret.rolling(10).std()
    merged["vol_20"] = ret.rolling(20).std()
    merged["vol_50"] = ret.rolling(50).std()
    merged["ewma_vol"] = ret.ewm(span=20).std()
    merged["sp_slope_20"] = merged["sp500_close"].diff(20)

    merged["quarter"] = merged["Date"].dt.quarter.astype("Int64")
    q = merged["quarter"].astype(float)
    merged["q_sin"] = np.sin(2*np.pi*(q-1)/4)
    merged["q_cos"] = np.cos(2*np.pi*(q-1)/4)
    qdum = pd.get_dummies(merged["quarter"], prefix="q", dtype=int)
    merged = pd.concat([merged, qdum], axis=1)

    # ======================
    # 5) Snapshot fundamentales/FRED al END_TRAIN
    # ======================
    snap = fundamentals_snapshot_at(pd.to_datetime(END_TRAIN))
    if not snap.empty:
        merged = pd.merge_asof(
            merged.sort_values("Date"),
            snap.sort_values("Date"),
            on="Date",
            direction="backward"
        )
        for c in snap.columns:
            if c != "Date" and c in merged.columns:
                merged[c] = merged[c].ffill()

    # ======================
    # 6) Noticias + FinBERT (rango forward)
    # ======================
    print("📰 Buscando noticias para FORWARD…")
    news_fwd, src_fwd = _collect_news_with_fallback(fwd_start, fwd_end)
    daily_news = _score_finbert_daily_weighted(news_fwd) if not news_fwd.empty else pd.DataFrame()
    if not daily_news.empty:
        merged = pd.merge_asof(
            merged.sort_values("Date"),
            daily_news.sort_values("Date"),
            on="Date",
            direction="backward"
        )
    else:
        for c, v in [
            ("neg", 0.0), ("neu", 1.0), ("pos", 0.0),
            ("news_count", 0.0),
            ("neg_ewm3", 0.0), ("neu_ewm3", 1.0), ("pos_ewm3", 0.0),
            ("news_pos_interact", 0.0),
        ]:
            merged[c] = v
    print(f"✅ Sentimiento integrado (forward) desde: {src_fwd}")

    # ======================
    # 7) Lags + selección de features para cada periodo
    # ======================
    all_cols = [c for c in merged.columns if c != "Date"]
    feats_lag1 = _lag_cols(merged, all_cols, lag=1)
    df_all = pd.concat([merged[["Date"]], feats_lag1], axis=1)

    X_fwd = df_all.set_index("Date").loc[fwd_start:fwd_end].reset_index()

    def _select_feats(df, feats_required):
        cols = ["Date"] + [c for c in df.columns if c != "Date" and c in feats_required]
        X = df.reindex(columns=cols, fill_value=np.nan)
        return X.loc[:, ~X.columns.duplicated()]

    X_p1 = _select_feats(X_fwd, FEATS_P1)
    X_p2 = _select_feats(X_fwd, FEATS_P2)

    proba1 = _predict_proba_calibrated(X_p1.drop(columns=["Date"]), art_p1)
    proba2 = _predict_proba_calibrated(X_p2.drop(columns=["Date"]), art_p2)
    proba_w = _weighted_average(proba1, proba2, w1, w2)

    X_meta = np.c_[proba1, proba2]
    proba_s = meta_clf.predict_proba(X_meta)[:, 1]

    D = pd.to_datetime(X_p1["Date"])

    pred_p1 = (proba1 >= thr1).astype(int)
    pred_p2 = (proba2 >= thr2).astype(int)
    pred_w  = (proba_w >= thr_weighted).astype(int)
    pred_s  = (proba_s >= thr_meta).astype(int)

    # ======================
    # 8) Target forward OOS (EMA40 + slope_10, t+1)
    # ======================
    close_series  = merged.set_index("Date")["close"]
    ema40_series  = merged.set_index("Date")["ema40"]
    slope10_series = merged.set_index("Date")["slope_10"]

    bull_t = ((slope10_series > 0) & (close_series > ema40_series)).astype(int)
    y = (
        bull_t.shift(-1)
        .loc[D.min():D.max()]
        .reindex(D)
        .fillna(0)
        .astype(int)
        .values
    )

    # ======================
    # 9) 9) Regla ADX DINÁMICA (mejorada) ===
    # ======================
    # ADX_14 lag1 alineado con las fechas forward (D)
    adx_lag1 = (
        merged.set_index("Date")["ADX_14"]
        .shift(1)               # usamos ADX hasta el cierre del día anterior
        .reindex(D)             # re-alineamos al índice D
        .values
    )

    # Construimos la señal dinámica + zona muerta
    pred_dyn, no_trade = build_dynamic_signal(
        proba_s=proba_s,
        adx_lag1=adx_lag1,
        thr_strong=0.50,   # tendencia muy fuerte: Bull si p>=0.50
        thr_mid_low=0.45,  # inicio zona muerta moderada
        thr_mid_high=0.70, # Bull en ADX 25–50 sólo si p>=0.70
        thr_flat_low=0.50, # inicio zona muerta en lateral
        thr_flat_high=0.80 # Bull en lateral sólo si p>=0.80
    )

    # Métricas de la señal dinámica
    f1_dyn = f1_score(y, pred_dyn, zero_division=0)
    acc_dyn = accuracy_score(y, pred_dyn)
    cm_dyn = confusion_matrix(y, pred_dyn)

    print(f"\n📈 Dynamic ADX (mejorado) FORWARD ({D.min().date()} → {D.max().date()}), N={len(y)}")
    print(f"F1 dinámico: {f1_dyn:.4f} | ACC dinámico: {acc_dyn:.4f}")
    print("Matriz de confusión (umbral dinámico mejorado):")
    print(cm_dyn)
    print(f"Días en zona muerta (no_trade_zone=1): {int(no_trade.sum())} de {len(no_trade)}")
    # =============================================

    # ======================
    # 10) Report métricas y guardar CSV
    # ======================
    def _report(tag, y_true, pred):
        f1 = f1_score(y_true, pred, zero_division=0) if len(np.unique(y_true))>1 else float("nan")
        acc = accuracy_score(y_true, pred)
        print(f"\n📈 {tag} FORWARD ({D.min().date()} → {D.max().date()}), N={len(y_true)}")
        print(f"F1={f1:.4f} | ACC={acc:.4f}")
        print(confusion_matrix(y_true, pred))
        print(classification_report(y_true, pred, digits=4, zero_division=0))
        return f1, acc

    _report("Period p1", y, pred_p1)
    _report("Period p2", y, pred_p2)
    _report("Weighted",  y, pred_w)
    _report("Stacking",  y, pred_s)
    _report("Dynamic ADX", y, pred_dyn)
    close_on_D = merged.set_index("Date")["close"].reindex(D).values

    out = pd.DataFrame({
        "Date": D,
        "close": close_on_D,   # <- CLAVE
        "y_true": y,
        "proba_p1": proba1,
        "pred_p1": pred_p1,
        "proba_p2": proba2,
        "pred_p2": pred_p2,
        "proba_weighted": proba_w,
        "pred_weighted": pred_w,
        "proba_stacking": proba_s,
        "pred_stacking": pred_s,
        "ADX_14_lag1": adx_lag1,
        "pred_dyn_ADX": pred_dyn,
        "no_trade_zone": no_trade,
    })

    fname = FORWARD_RESULTS_FILE_TEMPLATE.format(
    ticker=ticker,
    start=D.min().date(),
    end=D.max().date()
    )

    out.to_csv(fname, index=False)
    print("\n🔎 Detalle forward dinámico:")
    print(out.to_string(index=False))
    print("\n💾 Guardado:", fname)


        # Construimos la señal dinámica + zona muerta
    pred_dyn, no_trade = build_dynamic_signal(
        proba_s=proba_s,
        adx_lag1=adx_lag1,
        thr_strong=0.50,
        thr_mid_low=0.45,
        thr_mid_high=0.70,
        thr_flat_low=0.50,
        thr_flat_high=0.80
    )


    out.to_csv(FORWARD_RESULTS_FILE_TEMPLATE, index=False)
    print("\n🔎 Detalle forward:")
    print(out.to_string(index=False))
    print("\n💾 Guardado:", FORWARD_RESULTS_FILE_TEMPLATE)


# =============================================
#  Main
# =============================================
if __name__ == "__main__":
    # Entrena y calibra hasta 2025-08-31 (con target EMA40+slope_10 + noticias en todo el histórico)
    summary = train_two_models_and_ensemble(ticker="NVDA", start_date="2000-01-01")
    # Prueba out-of-sample del primero de 1 set al 12 de dic (con noticias de set obt nov dic)
    run_forward_eval_nov(ticker="NVDA")


🧠 Device Torch: cpu
🔄 Cargando FinBERT (esto puede tardar unos segundos)...
✅ FinBERT cargado
📡 Descargando datos diarios de NVDA desde FMP... 2000-01-01 → 2025-08-31
✅ 6454 días 2000-01-03 → 2025-08-29
🏦 Fundamentales FMP...
✅ Fundamentales: 73 filas, 374 cols
📊 S&P500 local (sp500.csv)...
💵 Treasury (rf)...
🔗 Merge total...
📈 Indicadores técnicos...
📰 Buscando noticias para ENTRENAMIENTO (máxima cobertura)…
✅ Sentimiento integrado (train) desde: finnhub — días con noticias: 163
📈 Features + target (lag1, anti-leak)...
✅ Dataset final: 6453 filas, 432 features (lag1).

📊 Bull-rate (t+1) por trimestre:
 quarter  bull_rate_next_day
       1            0.468632
       2            0.480488
       3            0.479141
       4            0.535557

🗓️ Entrenando periodo 2000_2019: 2000-01-01 → 2019-12-31


🔁 Modelos 2000_2019:  25%|██▌       | 1/4 [00:14<00:43, 14.47s/it]

   → RandomForest F1-CV: 0.8230


🔁 Modelos 2000_2019:  50%|█████     | 2/4 [00:30<00:30, 15.14s/it]

   → XGBoost      F1-CV: 0.7819


...

   → LightGBM     F1-CV: 0.7530
🏆 Mejor 2020_2025: RandomForest (F1-CV=0.8069)


c:\Users\Fabrizio\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


💾 Guardado NVDA_2020_2025_model.joblib (thr=0.667)
🧱 Meta-Logistic F1-CV (rolling): 0.8488
💾 Guardado NVDA_stack.joblib (thr_meta=0.390)
🎯 Period-Vote HOLDOUT → F1=0.8681, ACC=0.8590
[[510  77]
 [105 599]]
              precision    recall  f1-score   support

           0     0.8293    0.8688    0.8486       587
           1     0.8861    0.8509    0.8681       704

    accuracy                         0.8590      1291
   macro avg     0.8577    0.8598    0.8584      1291
weighted avg     0.8603    0.8590    0.8592      1291

🎯 Weighted HOLDOUT → F1=0.8629, ACC=0.8536
[[507  80]
 [109 595]]
              precision    recall  f1-score   support

           0     0.8231    0.8637    0.8429       587
           1     0.8815    0.8452    0.8629       704

    accuracy                         0.8536      1291
   macro avg     0.8523    0.8544    0.8529      1291
weighted avg     0.8549    0.8536    0.8538      1291

🎯 Stacking HOLDOUT → F1=0.8342, ACC=0.8079
[[419 168]
 [ 80 624]]
        

C:\Users\Fabrizio\AppData\Local\Temp\ipykernel_14404\4005738736.py:274: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return base.fillna(method="ffill").fillna(method="bfill")
C:\Users\Fabrizio\AppData\Local\Temp\ipykernel_14404\4005738736.py:274: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return base.fillna(method="ffill").fillna(method="bfill")


📰 Buscando noticias para FORWARD…
✅ Sentimiento integrado (forward) desde: finnhub

📈 Dynamic ADX (mejorado) FORWARD (2025-09-02 → 2025-12-11), N=72
F1 dinámico: 0.5902 | ACC dinámico: 0.6528
Matriz de confusión (umbral dinámico mejorado):
[[29 11]
 [14 18]]
Días en zona muerta (no_trade_zone=1): 8 de 72

📈 Period p1 FORWARD (2025-09-02 → 2025-12-11), N=72
F1=0.7273 | ACC=0.7083
[[23 17]
 [ 4 28]]
              precision    recall  f1-score   support

           0     0.8519    0.5750    0.6866        40
           1     0.6222    0.8750    0.7273        32

    accuracy                         0.7083        72
   macro avg     0.7370    0.7250    0.7069        72
weighted avg     0.7498    0.7083    0.7047        72


📈 Period p2 FORWARD (2025-09-02 → 2025-12-11), N=72
F1=0.6032 | ACC=0.6528
[[28 12]
 [13 19]]
              precision    recall  f1-score   support

           0     0.6829    0.7000    0.6914        40
           1     0.6129    0.5938    0.6032        32

    accuracy 

## 2. Evaluación forward y estrategias de trading

Sobre el período *forward* (out-of-sample, sep–dic 2025) se calcula el retorno real a `t+1` y se comparan dos estrategias:
- **Stacking puro** — opera según la predicción del meta-modelo.
- **Stacking + ADX + zona muerta** — filtra señales con el indicador de fuerza de tendencia (ADX) y una banda neutral para no operar en mercados sin dirección clara.

Se reportan win-rate, retorno medio, Sharpe, max drawdown y la *equity curve*.

In [11]:
import pandas as pd
import numpy as np

# =============================
# 1) Cargar CSV forward
# =============================
PATH = "NVDA_forward_2025-09-01_to_2025-12-11.csv"
df = pd.read_csv(PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =============================
# 2) Retorno diario real (t+1)
# =============================
df["ret_1d"] = df["close"].pct_change().shift(-1)
df["ret_1d"].fillna(0.0, inplace=True)

# =============================
# 3) Estrategias
# =============================

# A) Stacking puro
df["ret_stacking"] = df["pred_stacking"] * df["ret_1d"]

# B) Stacking + ADX + zona muerta
df["signal_adx"] = np.where(
    (df["pred_dyn_ADX"] == 1) & (df["no_trade_zone"] == 0),
    1,
    0
)
df["ret_stacking_adx"] = df["signal_adx"] * df["ret_1d"]

# =============================
# 4) Métricas
# =============================
def compute_metrics(returns: pd.Series, name: str):
    returns = returns.copy()
    trades = returns[returns != 0]

    win_rate = (trades > 0).mean() if len(trades) > 0 else 0.0
    avg_return = trades.mean() if len(trades) > 0 else 0.0

    equity = (1 + returns).cumprod()
    peak = equity.cummax()
    drawdown = (equity / peak) - 1
    max_dd = drawdown.min()

    sharpe = (
        np.sqrt(252) * returns.mean() / returns.std()
        if returns.std() > 0 else 0.0
    )

    return {
        "strategy": name,
        "trades": int((returns != 0).sum()),
        "win_rate": round(win_rate, 4),
        "avg_return": round(avg_return, 5),
        "max_drawdown": round(max_dd, 4),
        "sharpe": round(sharpe, 3),
        "total_return": round(equity.iloc[-1] - 1, 4)
    }

# =============================
# 5) Comparación
# =============================
results = pd.DataFrame([
    compute_metrics(df["ret_stacking"], "Stacking puro"),
    compute_metrics(df["ret_stacking_adx"], "Stacking + ADX")
])

print("\n📊 RESULTADOS P&L SIMULADO\n")
print(results.to_string(index=False))

# =============================
# 6) Equity curve (opcional)
# =============================
df["equity_stacking"] = (1 + df["ret_stacking"]).cumprod()
df["equity_stacking_adx"] = (1 + df["ret_stacking_adx"]).cumprod()

df_out = df[[
    "Date",
    "ret_1d",
    "pred_stacking",
    "signal_adx",
    "ret_stacking",
    "ret_stacking_adx",
    "equity_stacking",
    "equity_stacking_adx"
]]

df_out.to_csv("NVDA_pnl_comparison.csv", index=False)
print("\n💾 Guardado: NVDA_pnl_comparison.csv")



📊 RESULTADOS P&L SIMULADO

      strategy  trades  win_rate  avg_return  max_drawdown  sharpe  total_return
 Stacking puro      52    0.6154     0.00628       -0.1053   2.638        0.3491
Stacking + ADX      44    0.6364     0.00758       -0.0700   2.994        0.3655

💾 Guardado: NVDA_pnl_comparison.csv


## 3. Orquestación: entrenar (si falta) → forward → P&L

Celda de control: entrena solo si faltan los artefactos, ejecuta siempre la evaluación forward,
localiza el CSV generado y calcula el P&L de la estrategia a partir de él.

In [13]:
import os
import glob
import pandas as pd

TICKERS = ["NVDA", "AAPL", "MSFT", "AMD", "META"]  # poné los que quieras

def artifacts_exist(ticker: str) -> bool:
    need = [
        f"{ticker}_2000_2019_model.joblib",
        f"{ticker}_2020_2025_model.joblib",
        f"{ticker}_stack.joblib",
        f"{ticker}_ensemble.json",
    ]
    return all(os.path.exists(x) for x in need)

def find_latest_forward_csv(ticker: str) -> str | None:
    # busca el más reciente que matchee el patrón real
    patt = f"{ticker}_forward_*_to_*_ADXdyn.csv"
    files = sorted(glob.glob(patt))
    return files[-1] if files else None

summary_rows = []

for t in TICKERS:
    print("\n" + "="*80)
    print(f"🚀 Ticker: {t}")

    # 1) Entrenar sólo si falta
    if not artifacts_exist(t):
        print("🧠 No hay artefactos. Entrenando…")
        train_two_models_and_ensemble(ticker=t, start_date="2000-01-01")
    else:
        print("✅ Artefactos encontrados. Salteo entrenamiento.")

    # 2) Forward (siempre)
    run_forward_eval_nov(ticker=t)

    # 3) Ubicar el CSV guardado
    fwd_path = find_latest_forward_csv(t)
    if not fwd_path:
        print(f"⚠️ No encontré forward CSV para {t}.")
        continue

    # 4) P&L desde el forward CSV
    res = pnl_from_forward_csv(fwd_path)   # <- tu función de P&L
    for _, r in res.iterrows():
        summary_rows.append({
            "ticker": t,
            "strategy": r["strategy"],
            "trades": r["trades"],
            "win_rate": r["win_rate"],
            "avg_return": r["avg_return"],
            "max_drawdown": r["max_drawdown"],
            "sharpe": r["sharpe"],
            "total_return": r["total_return"],
            "forward_file": fwd_path,
        })

summary = pd.DataFrame(summary_rows).sort_values(["ticker","strategy"])
print("\n📌 RESUMEN MULTI-TICKER")
print(summary.to_string(index=False))
summary.to_csv("multi_ticker_pnl_summary.csv", index=False)
print("\n💾 Guardado: multi_ticker_pnl_summary.csv")



🚀 Ticker: NVDA
✅ Artefactos encontrados. Salteo entrenamiento.


C:\Users\Fabrizio\AppData\Local\Temp\ipykernel_14404\4005738736.py:274: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return base.fillna(method="ffill").fillna(method="bfill")
C:\Users\Fabrizio\AppData\Local\Temp\ipykernel_14404\4005738736.py:274: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return base.fillna(method="ffill").fillna(method="bfill")


📰 Buscando noticias para FORWARD…
✅ Sentimiento integrado (forward) desde: finnhub

📈 Dynamic ADX (mejorado) FORWARD (2025-09-02 → 2025-12-11), N=72
F1 dinámico: 0.5902 | ACC dinámico: 0.6528
Matriz de confusión (umbral dinámico mejorado):
[[29 11]
 [14 18]]
Días en zona muerta (no_trade_zone=1): 8 de 72

📈 Period p1 FORWARD (2025-09-02 → 2025-12-11), N=72
F1=0.7273 | ACC=0.7083
[[23 17]
 [ 4 28]]
              precision    recall  f1-score   support

           0     0.8519    0.5750    0.6866        40
           1     0.6222    0.8750    0.7273        32

    accuracy                         0.7083        72
   macro avg     0.7370    0.7250    0.7069        72
weighted avg     0.7498    0.7083    0.7047        72


📈 Period p2 FORWARD (2025-09-02 → 2025-12-11), N=72
F1=0.6032 | ACC=0.6528
[[28 12]
 [13 19]]
              precision    recall  f1-score   support

           0     0.6829    0.7000    0.6914        40
           1     0.6129    0.5938    0.6032        32

    accuracy 

🔁 Modelos 2000_2019:  25%|██▌       | 1/4 [00:16<00:48, 16.13s/it]

   → RandomForest F1-CV: 0.8430


...

   → LightGBM     F1-CV: 0.7137
🏆 Mejor 2020_2025: RandomForest (F1-CV=0.7542)


c:\Users\Fabrizio\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


💾 Guardado META_2020_2025_model.joblib (thr=0.818)
🧱 Meta-Logistic F1-CV (rolling): 0.8433
💾 Guardado META_stack.joblib (thr_meta=0.631)
🎯 Period-Vote HOLDOUT → F1=0.9034, ACC=0.8757
[[197  42]
 [ 41 388]]
              precision    recall  f1-score   support

           0     0.8277    0.8243    0.8260       239
           1     0.9023    0.9044    0.9034       429

    accuracy                         0.8757       668
   macro avg     0.8650    0.8643    0.8647       668
weighted avg     0.8756    0.8757    0.8757       668

🎯 Weighted HOLDOUT → F1=0.8967, ACC=0.8683
[[198  41]
 [ 47 382]]
              precision    recall  f1-score   support

           0     0.8082    0.8285    0.8182       239
           1     0.9031    0.8904    0.8967       429

    accuracy                         0.8683       668
   macro avg     0.8556    0.8594    0.8574       668
weighted avg     0.8691    0.8683    0.8686       668

🎯 Stacking HOLDOUT → F1=0.8886, ACC=0.8563
[[189  50]
 [ 46 383]]
        

C:\Users\Fabrizio\AppData\Local\Temp\ipykernel_14404\4005738736.py:274: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return base.fillna(method="ffill").fillna(method="bfill")
C:\Users\Fabrizio\AppData\Local\Temp\ipykernel_14404\4005738736.py:274: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return base.fillna(method="ffill").fillna(method="bfill")


📰 Buscando noticias para FORWARD…
✅ Sentimiento integrado (forward) desde: finnhub

📈 Dynamic ADX (mejorado) FORWARD (2025-09-02 → 2025-12-11), N=72
F1 dinámico: 0.6364 | ACC dinámico: 0.7778
Matriz de confusión (umbral dinámico mejorado):
[[42  9]
 [ 7 14]]
Días en zona muerta (no_trade_zone=1): 1 de 72

📈 Period p1 FORWARD (2025-09-02 → 2025-12-11), N=72
F1=0.6222 | ACC=0.7639
[[41 10]
 [ 7 14]]
              precision    recall  f1-score   support

           0     0.8542    0.8039    0.8283        51
           1     0.5833    0.6667    0.6222        21

    accuracy                         0.7639        72
   macro avg     0.7188    0.7353    0.7253        72
weighted avg     0.7752    0.7639    0.7682        72


📈 Period p2 FORWARD (2025-09-02 → 2025-12-11), N=72
F1=0.6364 | ACC=0.7778
[[42  9]
 [ 7 14]]
              precision    recall  f1-score   support

           0     0.8571    0.8235    0.8400        51
           1     0.6087    0.6667    0.6364        21

    accuracy 

## 4. Construcción del portafolio (risk parity)

Combina las señales de los 5 tickers en un portafolio con *risk parity* simple (pesos ∝ 1/volatilidad
out-of-sample), armando los retornos diarios de cada estrategia desde los CSV forward.

In [2]:
import os
import glob
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
TICKERS = ["NVDA", "AAPL", "MSFT", "AMD", "META"]

# Elegí qué estrategia usar para portfolio:
#   "Stacking puro" o "Stacking + ADX"
PORTFOLIO_STRATEGY = "Stacking + ADX"

# Si tu forward no tiene close, vamos a usar un proxy:
# ret_1d_proxy = +1 si y_true(t+1)=1, -1 si y_true(t+1)=0
# (esto NO es retorno real de precio, pero permite comparar señales “como clasificador”)
USE_PROXY_RETURNS_IF_NO_CLOSE = True

# Risk parity simple: pesos ~ 1/vol (vol OOS)
ANNUALIZATION = 252


# =========================
# HELPERS: archivos
# =========================
def find_latest_forward_csv(ticker: str) -> str | None:
    patt = f"{ticker}_forward_*_to_*_ADXdyn.csv"
    files = sorted(glob.glob(patt))
    return files[-1] if files else None


# =========================
# HELPERS: construir retornos diarios desde forward CSV
# =========================
def add_returns_from_forward(df: pd.DataFrame) -> pd.DataFrame:
    """
    Necesita columnas:
      Date
      pred_stacking
      pred_dyn_ADX
      no_trade_zone
    Idealmente:
      close  -> para retorno real t+1
    Alternativa proxy:
      y_true -> para retorno proxy (+1/-1)
    """

    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    # 1) Retorno base (t+1)
    if "close" in df.columns:
        # retorno real (t+1)
        df["ret_1d"] = df["close"].pct_change().shift(-1)
        df["ret_1d"] = df["ret_1d"].fillna(0.0)
        ret_type = "price"
    else:
        if not USE_PROXY_RETURNS_IF_NO_CLOSE:
            raise KeyError("El forward CSV no tiene 'close'. Seteá USE_PROXY_RETURNS_IF_NO_CLOSE=True o agregá close.")
        if "y_true" not in df.columns:
            raise KeyError("No hay 'close' ni 'y_true' para construir retornos. Necesito al menos uno.")
        # proxy: si mañana es bull (y_true=1) => +1bp, si no => -1bp (ejemplo simple)
        # podés cambiar el tamaño del paso: 0.001 = 0.10%
        step = 0.001
        df["ret_1d"] = np.where(df["y_true"].astype(int) == 1, step, -step)
        ret_type = "proxy"

    # 2) Estrategia A: Stacking puro (entra si pred_stacking=1)
    if "pred_stacking" not in df.columns:
        raise KeyError("Falta 'pred_stacking' en el forward CSV.")
    df["ret_stacking"] = df["pred_stacking"].astype(int) * df["ret_1d"]

    # 3) Estrategia B: Stacking + ADX (entra si pred_dyn_ADX=1 y no_trade_zone=0)
    if "pred_dyn_ADX" not in df.columns or "no_trade_zone" not in df.columns:
        raise KeyError("Faltan 'pred_dyn_ADX' y/o 'no_trade_zone' en el forward CSV.")
    signal_adx = ((df["pred_dyn_ADX"].astype(int) == 1) & (df["no_trade_zone"].astype(int) == 0)).astype(int)
    df["ret_stacking_adx"] = signal_adx * df["ret_1d"]

    df.attrs["ret_type"] = ret_type
    return df
# =========================
# 1. Extraer la tasa más reciente
def fetch_treasury_10y(api_key):
        url = f"https://financialmodelingprep.com/stable/treasury-rates?apikey={api_key}"
        t = requests.get(url, timeout=25).json()
        dt = pd.DataFrame(t)
        dt["Date"] = pd.to_datetime(dt["date"], errors="coerce")
        dt = dt.sort_values("Date")
        ten_cols = [c for c in dt.columns if "10" in c and ("year" in c.lower() or "y" in c.lower())]
        if ten_cols:
            dt["rf"] = pd.to_numeric(dt[ten_cols[0]], errors="coerce") / 100.0
        else:
            numc = dt.select_dtypes(include=[np.number]).columns
            dt["rf"] = (dt[numc].mean(axis=1) / 100.0) if len(numc) else 0.0
        return dt[["Date", "rf"]]

import requests
FMP_API_KEY = os.getenv("FMP_API_KEY", "")
treas  = fetch_treasury_10y(FMP_API_KEY)
rf_anual = treas["rf"].iloc[-1] if not treas.empty else 0.04 # 4% fallback
rf_diaria = rf_anual / 252

def compute_metrics(returns: pd.Series) -> dict:
    returns = returns.fillna(0.0).copy()
    trades = returns[returns != 0]

    win_rate = float((trades > 0).mean()) if len(trades) > 0 else 0.0
    avg_return = float(trades.mean()) if len(trades) > 0 else 0.0

    equity = (1 + returns).cumprod()
    peak = equity.cummax()
    dd = (equity / peak) - 1
    max_dd = float(dd.min()) if len(dd) else 0.0

    vol = float(returns.std()) if returns.std() > 0 else 0.0
    

    # 2. Calcular Sharpe (asumiendo que 'returns' es una Serie de Pandas con retornos diarios)
   
    sharpe = float(np.sqrt(ANNUALIZATION) * (returns.mean() - rf_diaria) / returns.std()) if returns.std() > 0 else 0.0
    total_return = float(equity.iloc[-1] - 1) if len(equity) else 0.0

    return {
        "trades": int((returns != 0).sum()),
        "win_rate": win_rate,
        "avg_return": avg_return,
        "vol_daily": vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "total_return": total_return,
    }


# =========================
# 1) Cargar resumen multi-ticker si existe
# =========================
SUMMARY_FILE = "multi_ticker_pnl_summary.csv"
if os.path.exists(SUMMARY_FILE):
    summary = pd.read_csv(SUMMARY_FILE)
    print("\n✅ Cargado:", SUMMARY_FILE)
else:
    summary = pd.DataFrame()
    print("\n⚠️ No encontré multi_ticker_pnl_summary.csv. Igual puedo armarlo leyendo los forward CSV.")


# =========================
# 2) Leer forward CSVs, armar tabla por ticker y estrategia
# =========================
rows = []
daily_returns = {}  # (ticker, strategy_name) -> Series index Date

ret_types = set()

for t in TICKERS:
    path = find_latest_forward_csv(t)
    if not path:
        print(f"⚠️ No encontré forward CSV para {t}. Saltando.")
        continue

    df_fwd = pd.read_csv(path, parse_dates=["Date"])
    df_fwd = add_returns_from_forward(df_fwd)
    ret_types.add(df_fwd.attrs.get("ret_type", "unknown"))

    # guardo retornos diarios
    sA = df_fwd.set_index("Date")["ret_stacking"]
    sB = df_fwd.set_index("Date")["ret_stacking_adx"]
    daily_returns[(t, "Stacking puro")] = sA
    daily_returns[(t, "Stacking + ADX")] = sB

    # métricas por estrategia
    mA = compute_metrics(sA)
    mB = compute_metrics(sB)

    rows.append({"ticker": t, "strategy": "Stacking puro", **mA, "forward_file": path})
    rows.append({"ticker": t, "strategy": "Stacking + ADX", **mB, "forward_file": path})

metrics_tbl = pd.DataFrame(rows)
if metrics_tbl.empty:
    raise RuntimeError("No pude construir métricas. Revisá que existan forward CSVs y que tengan columnas necesarias.")

# Formateo amigable
show_tbl = metrics_tbl.copy()
for c in ["win_rate", "avg_return", "vol_daily", "sharpe", "max_drawdown", "total_return"]:
    show_tbl[c] = show_tbl[c].astype(float)

print("\n📊 TABLA DE RESULTADOS (por ticker y estrategia)")
print(show_tbl.sort_values(["ticker", "strategy"]).to_string(index=False))


# =========================
# 3) Ranking Sharpe OOS (para la estrategia elegida en portfolio)
# =========================
sel = metrics_tbl[metrics_tbl["strategy"] == PORTFOLIO_STRATEGY].copy()
sel = sel.sort_values("sharpe", ascending=False)
sel["rank_sharpe"] = np.arange(1, len(sel) + 1)

print(f"\n🏆 RANKING por Sharpe OOS — Estrategia: {PORTFOLIO_STRATEGY}")
print(sel[["ticker", "sharpe", "vol_daily", "max_drawdown", "total_return", "trades", "rank_sharpe"]].to_string(index=False))


# =========================
# 4) Portfolio equal-risk (risk parity simple: pesos ~ 1/vol)
# =========================
# pesos ~ 1/vol, normalizados a 1
sel = sel.replace([np.inf, -np.inf], np.nan).dropna(subset=["vol_daily"])
sel = sel[sel["vol_daily"] > 0].copy()

if sel.empty:
    raise RuntimeError("No hay vol_daily > 0 para construir risk parity. (¿retornos todos 0?)")

sel["w_raw"] = 1.0 / sel["vol_daily"]
sel["weight"] = sel["w_raw"] / sel["w_raw"].sum()

weights = sel.set_index("ticker")["weight"].to_dict()

print("\n⚖️ PESOS equal-risk (1/vol) — usando vol OOS diaria")
wdf = sel[["ticker", "vol_daily", "sharpe", "weight"]].sort_values("weight", ascending=False)
print(wdf.to_string(index=False))


# =========================
# 5) Backtest agregado (portfolio)
# =========================
# Alineamos series por fecha y armamos retorno diario del portfolio
port_rets = None

for t in sel["ticker"].tolist():
    s = daily_returns[(t, PORTFOLIO_STRATEGY)].copy()
    s.name = t
    if port_rets is None:
        port_rets = pd.DataFrame(s)
    else:
        port_rets = port_rets.join(s, how="outer")

port_rets = port_rets.fillna(0.0)

# retorno diario portfolio = sum_i w_i * r_i
W = pd.Series(weights).reindex(port_rets.columns).fillna(0.0)
portfolio_ret = port_rets.mul(W, axis=1).sum(axis=1)
portfolio_ret.name = "portfolio_ret"

pm = compute_metrics(portfolio_ret)

equity = (1 + portfolio_ret).cumprod()
out_port = pd.DataFrame({
    "Date": equity.index,
    "portfolio_ret": portfolio_ret.values,
    "portfolio_equity": equity.values,
})
out_port.to_csv("portfolio_backtest_aggregated.csv", index=False)

print("\n📈 BACKTEST AGREGADO — Portfolio equal-risk")
print(f"Ret type: {list(ret_types)} (si aparece 'proxy', NO es precio real)")
print(pd.DataFrame([{
    "strategy": f"PORTFOLIO ({PORTFOLIO_STRATEGY})",
    **pm,
}]).to_string(index=False))

print("\n💾 Guardado: portfolio_backtest_aggregated.csv")
print("💾 Guardado: (opcional) podés guardar weights en otro CSV si querés.")


# =========================
# 6) Guardar tablas finales
# =========================
metrics_tbl.to_csv("per_ticker_strategy_metrics.csv", index=False)
wdf.to_csv("portfolio_weights_equal_risk.csv", index=False)

print("\n💾 Guardado: per_ticker_strategy_metrics.csv")
print("💾 Guardado: portfolio_weights_equal_risk.csv")



✅ Cargado: multi_ticker_pnl_summary.csv

📊 TABLA DE RESULTADOS (por ticker y estrategia)
ticker       strategy  trades  win_rate  avg_return  vol_daily    sharpe  max_drawdown  total_return                                     forward_file
  AAPL Stacking + ADX      56  0.500000    0.001076   0.012176  0.875835     -0.063298      0.056539 AAPL_forward_2025-09-02_to_2025-12-11_ADXdyn.csv
  AAPL  Stacking puro      51  0.450980    0.000481   0.011898  0.234171     -0.063298      0.019716 AAPL_forward_2025-09-02_to_2025-12-11_ADXdyn.csv
   AMD Stacking + ADX      24  0.541667    0.015862   0.036066  2.254598     -0.087706      0.401893  AMD_forward_2025-09-02_to_2025-12-11_ADXdyn.csv
   AMD  Stacking puro      48  0.520833    0.005981   0.038319  1.583392     -0.196321      0.269395  AMD_forward_2025-09-02_to_2025-12-11_ADXdyn.csv
  META Stacking + ADX      22  0.454545   -0.005023   0.015465 -1.744763     -0.147716     -0.112720 META_forward_2025-09-02_to_2025-12-11_ADXdyn.csv
  META  St

## 5. Portafolio equal-risk y backtest agregado

Selecciona los tickers con Sharpe positivo, aplica los pesos equal-risk y genera el backtest agregado
(`results/backtest/portfolio_backtest_*.csv`) junto con la *equity curve*.

In [3]:
# Portfolio equal-risk SOLO con Sharpe positivo
STRAT = "Stacking + ADX"

sel = metrics_tbl[metrics_tbl["strategy"] == STRAT].copy()
sel = sel[sel["sharpe"] > 0].copy()          # <-- filtro clave
sel = sel[sel["vol_daily"] > 0].copy()

sel["w_raw"] = 1.0 / sel["vol_daily"]
sel["weight"] = sel["w_raw"] / sel["w_raw"].sum()
weights_pos = sel.set_index("ticker")["weight"].to_dict()

print(sel[["ticker","sharpe","vol_daily","weight"]].sort_values("weight", ascending=False))

# Backtest agregado con esos pesos
port_rets = None
for t in sel["ticker"].tolist():
    s = daily_returns[(t, STRAT)].copy()
    s.name = t
    port_rets = pd.DataFrame(s) if port_rets is None else port_rets.join(s, how="outer")

port_rets = port_rets.fillna(0.0)
W = pd.Series(weights_pos).reindex(port_rets.columns).fillna(0.0)

portfolio_ret_pos = port_rets.mul(W, axis=1).sum(axis=1)
equity_pos = (1 + portfolio_ret_pos).cumprod()

# métricas
pm_pos = compute_metrics(portfolio_ret_pos)
print("\nPORTFOLIO Sharpe>0")
print(pm_pos)

pd.DataFrame({
    "Date": equity_pos.index,
    "portfolio_ret": portfolio_ret_pos.values,
    "portfolio_equity": equity_pos.values,
}).to_csv("portfolio_backtest_sharpe_pos.csv", index=False)

print("\n💾 Guardado: portfolio_backtest_sharpe_pos.csv")


  ticker    sharpe  vol_daily    weight
3   AAPL  0.875835   0.012176  0.470849
1   NVDA  0.624004   0.015486  0.370193
7    AMD  2.254598   0.036066  0.158958

PORTFOLIO Sharpe>0
{'trades': 63, 'win_rate': 0.4603174603174603, 'avg_return': 0.001738232883351004, 'vol_daily': 0.011659787140828113, 'sharpe': 1.8459890127774277, 'max_drawdown': -0.05970279066669093, 'total_return': 0.11027198541970074}

💾 Guardado: portfolio_backtest_sharpe_pos.csv


## 6. Costos de transacción

Ajusta los retornos por costos (comisiones/slippage en *bps*), aplicados solo cuando hay cambio de
posición, para comparar el desempeño **bruto vs neto**.

In [4]:
def apply_transaction_costs(returns: pd.Series, signal: pd.Series, cost_bps=10):
    """
    returns: retornos diarios
    signal:  0/1 posición diaria
    cost_bps: costo ida+vuelta en basis points (10 = 0.10%)
    """
    cost = cost_bps / 10000.0

    # cambios de posición
    trades = signal.diff().abs().fillna(0)

    # costo solo cuando hay trade
    costs = trades * cost

    return returns - costs
# returns diarios agregados (ya los tenés)
ret_adx = portfolio_ret_pos.copy()        # stacking + ADX
sig_adx = (portfolio_ret_pos != 0).astype(int)

ret_adx_net = apply_transaction_costs(
    returns=ret_adx,
    signal=sig_adx,
    cost_bps=10
)
# armamos returns stacking puro con los mismos pesos
STRAT_PURE = "Stacking puro"

port_pure = None
for t in ["AAPL", "NVDA", "AMD"]:
    s = daily_returns[(t, STRAT_PURE)].copy()
    s.name = t
    port_pure = pd.DataFrame(s) if port_pure is None else port_pure.join(s, how="outer")

port_pure = port_pure.fillna(0.0)
ret_pure = port_pure.mul(W, axis=1).sum(axis=1)

sig_pure = (ret_pure != 0).astype(int)

ret_pure_net = apply_transaction_costs(
    returns=ret_pure,
    signal=sig_pure,
    cost_bps=10
)

metrics_pure_net = compute_metrics(ret_pure_net)
print("\nPORTFOLIO STACKING PURO + COSTOS")
print(metrics_pure_net)

metrics_adx_net = compute_metrics(ret_adx_net)
print("PORTFOLIO ADX + COSTOS")
print(metrics_adx_net)



PORTFOLIO STACKING PURO + COSTOS
{'trades': 67, 'win_rate': 0.43283582089552236, 'avg_return': 0.0009848577178065284, 'vol_daily': 0.012647741555819847, 'sharpe': 0.943083338149952, 'max_drawdown': -0.08716171655719385, 'total_return': 0.062130887364398646}
PORTFOLIO ADX + COSTOS
{'trades': 66, 'win_rate': 0.4393939393939394, 'avg_return': 0.001583464721986564, 'vol_daily': 0.011679333477184169, 'sharpe': 1.7485109585727407, 'max_drawdown': -0.05970279066669093, 'total_return': 0.10472471453404086}


## ✅ Conclusiones

- Los ensembles por régimen + stacking logran **F1 ≈ 0.85–0.90** en el holdout temporal.
- En el *forward* real los resultados son **mixtos** (AMD claramente positivo; otros planos/negativos),
  evidenciando la brecha entre precisión de clasificación y rentabilidad operativa neta.
- El análisis crítico completo y los próximos pasos están en el [`README`](../README.md).